In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pandas as pd
import numpy as np
from admet_ai import ADMETModel

In [2]:
#csv loading

df = pd.read_csv("orexin_ligand_library_raw.csv")
df_raw = df.copy()    # preserve the original — never modify this

In [3]:
#checking the df loaded properly; should get 80
print(len(df))

80


In [4]:
import requests
import time

def fetch_smiles_for_cid(cid):
    """Fetch SMILES directly from PubChem REST API for a given CID."""
    url = (
        f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/"
        f"{cid}/property/IsomericSMILES,CanonicalSMILES/JSON"
    )
    try:
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        props = (response.json()
                 .get("PropertyTable", {})
                 .get("Properties", [{}])[0])
        # Updated key names to match what PubChem actually returns
        return props.get("SMILES"), props.get("ConnectivitySMILES")
    except Exception as e:
        print(f"  Failed for CID {cid}: {e}")
        return None, None

print("Fetching missing SMILES for all 80 compounds...")
isomeric_list = []
canonical_list = []

for i, row in df.iterrows():
    iso, can = fetch_smiles_for_cid(row["cid"])
    isomeric_list.append(iso)
    canonical_list.append(can)
    time.sleep(0.2)
    if (i + 1) % 10 == 0:
        print(f"  Fetched {i + 1} / {len(df)}...")

df["isomeric_smiles"]  = isomeric_list
df["canonical_smiles"] = canonical_list

print(f"\nDone. SMILES missing after fetch:")
print(f"  isomeric_smiles:  {df['isomeric_smiles'].isnull().sum()}")
print(f"  canonical_smiles: {df['canonical_smiles'].isnull().sum()}")

df.to_csv("orexin_ligand_library_raw.csv", index=False)
print("\nRepaired file saved.")

Fetching missing SMILES for all 80 compounds...
  Fetched 10 / 80...
  Fetched 20 / 80...
  Fetched 30 / 80...
  Fetched 40 / 80...
  Fetched 50 / 80...
  Fetched 60 / 80...
  Fetched 70 / 80...
  Fetched 80 / 80...

Done. SMILES missing after fetch:
  isomeric_smiles:  0
  canonical_smiles: 0

Repaired file saved.


In [6]:
print(f"Total compounds: {len(df)}")
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nFirst few SMILES:")
print(df[["name", "isomeric_smiles"]].head(4))

Total compounds: 80

Missing values per column:
name                 0
drug_class           0
cid                  0
molecular_formula    0
molecular_weight     0
iupac_name           0
canonical_smiles     0
isomeric_smiles      0
xlogp                1
h_bond_donors        0
h_bond_acceptors     0
rotatable_bonds      0
tpsa                 0
exact_mass           0
charge               0
dtype: int64

First few SMILES:
           name                                    isomeric_smiles
0  Daridorexant  CC1=C(C=CC2=C1N=C(N2)[C@@]3(CCCN3C(=O)C4=C(C=C...
1   Lemborexant  CC1=NC(=NC=C1OC[C@]2(C[C@H]2C(=O)NC3=NC=C(C=C3...
2    Suvorexant  C[C@@H]1CCN(CCN1C(=O)C2=C(C=CC(=C2)C)N3N=CC=N3...
3   Oveporexton  CCS(=O)(=O)N[C@@H]1[C@@H](N(CC1(F)F)C(=O)C(C)(...


In [7]:
# Test what PubChem actually returns for one CID
import requests

cid = 91801202  # Daridorexant

url = (
    f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/"
    f"{cid}/property/IsomericSMILES,CanonicalSMILES/JSON"
)

response = requests.get(url, timeout=15)
print(f"Status code: {response.status_code}")
print(f"Raw response:")
print(response.text)

Status code: 200
Raw response:
{
  "PropertyTable": {
    "Properties": [
      {
        "CID": 91801202,
        "SMILES": "CC1=C(C=CC2=C1N=C(N2)[C@@]3(CCCN3C(=O)C4=C(C=CC(=C4)OC)N5N=CC=N5)C)Cl",
        "ConnectivitySMILES": "CC1=C(C=CC2=C1N=C(N2)C3(CCCN3C(=O)C4=C(C=CC(=C4)OC)N5N=CC=N5)C)Cl"
      }
    ]
  }
}



In [8]:
#confirm library loaded correctly (optional step)

print(f"Loaded {len(df)} compounds")
print(f"Columns: {list(df.columns)}")
df.head()

Loaded 80 compounds
Columns: ['name', 'drug_class', 'cid', 'molecular_formula', 'molecular_weight', 'iupac_name', 'canonical_smiles', 'isomeric_smiles', 'xlogp', 'h_bond_donors', 'h_bond_acceptors', 'rotatable_bonds', 'tpsa', 'exact_mass', 'charge']


,name,drug_class,cid,molecular_formula,molecular_weight,iupac_name,canonical_smiles,isomeric_smiles,xlogp,h_bond_donors,h_bond_acceptors,rotatable_bonds,tpsa,exact_mass,charge
0,Daridorexant,DORA,91801202,C23H23ClN6O2,450.9,[(2S)-2-(5-chloro-4-methyl-1H-benzimidazol-2-y...,CC1=C(C=CC2=C1N=C(N2)C3(CCCN3C(=O)C4=C(C=CC(=C...,CC1=C(C=CC2=C1N=C(N2)[C@@]3(CCCN3C(=O)C4=C(C=C...,4.1,1,5,4,88.9,450.157102,0
1,Lemborexant,DORA,56944144,C22H20F2N4O2,410.4,"cis-(1R,2S)-2-[(2,4-dimethylpyrimidin-5-yl)oxy...",CC1=NC(=NC=C1OCC2(CC2C(=O)NC3=NC=C(C=C3)F)C4=C...,CC1=NC(=NC=C1OC[C@]2(C[C@H]2C(=O)NC3=NC=C(C=C3...,3.2,1,7,6,77.0,410.155432,0
2,Suvorexant,DORA,24965990,C23H23ClN6O2,450.9,"[(7R)-4-(5-chloro-1,3-benzoxazol-2-yl)-7-methy...",CC1CCN(CCN1C(=O)C2=C(C=CC(=C2)C)N3N=CC=N3)C4=N...,C[C@@H]1CCN(CCN1C(=O)C2=C(C=CC(=C2)C)N3N=CC=N3...,4.9,0,6,3,80.3,450.157102,0
3,Oveporexton,agonist,154617563,C23H25F5N2O4S,520.5,"N-[(2S,3R)-2-[[3-(3,5-difluorophenyl)-2-fluoro...",CCS(=O)(=O)NC1C(N(CC1(F)F)C(=O)C(C)(C)O)CC2=C(...,CCS(=O)(=O)N[C@@H]1[C@@H](N(CC1(F)F)C(=O)C(C)(...,3.6,2,10,7,95.1,520.145519,0
4,similar_to_91801202_91809208,DORA_analog,91809208,C23H24Cl2N6O2,487.4,[(2S)-2-(5-chloro-4-methyl-1H-benzimidazol-2-y...,CC1=C(C=CC2=C1N=C(N2)C3(CCCN3C(=O)C4=C(C=CC(=C...,CC1=C(C=CC2=C1N=C(N2)[C@@]3(CCCN3C(=O)C4=C(C=C...,NaN,2,5,4,88.9,486.133779,0


In [9]:
# Examine the raw file before any cleaning
print(f"Total compounds: {len(df)}")
print(f"\nData types of each column:")
print(df.dtypes)
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nHow many have the text string 'None' in isomeric_smiles:")
print((df["isomeric_smiles"] == "None").sum())
print(f"\nHow many have the text string 'None' in molecular_weight:")
print((df["molecular_weight"] == "None").sum())
print(f"\nFirst few values of molecular_weight column:")
print(df["molecular_weight"].head(10).tolist())

Total compounds: 80

Data types of each column:
name                     str
drug_class               str
cid                    int64
molecular_formula        str
molecular_weight     float64
iupac_name               str
canonical_smiles         str
isomeric_smiles          str
xlogp                float64
h_bond_donors          int64
h_bond_acceptors       int64
rotatable_bonds        int64
tpsa                 float64
exact_mass           float64
charge                 int64
dtype: object

Missing values per column:
name                 0
drug_class           0
cid                  0
molecular_formula    0
molecular_weight     0
iupac_name           0
canonical_smiles     0
isomeric_smiles      0
xlogp                1
h_bond_donors        0
h_bond_acceptors     0
rotatable_bonds      0
tpsa                 0
exact_mass           0
charge               0
dtype: int64

How many have the text string 'None' in isomeric_smiles:
0

How many have the text string 'None' in molecular_weight

In [10]:
# ============================================================
# 2b. Basic cleaning
# ============================================================

# Convert numeric columns to numbers (sometimes imported as text)

df = df.dropna(subset=["isomeric_smiles"])
df["molecular_weight"] = pd.to_numeric(df["molecular_weight"], errors="coerce")
df["xlogp"]            = pd.to_numeric(df["xlogp"],            errors="coerce")
df["tpsa"]             = pd.to_numeric(df["tpsa"],             errors="coerce")
df = df[df["molecular_weight"].between(200, 1000)]
df = df.drop_duplicates(subset="cid")

print(f"After cleaning: {len(df)} compounds remain.")

After cleaning: 80 compounds remain.


In [11]:
# ============================================================
# 2c. Lipinski Rule of 5 — flag each violation
# ============================================================

# Each column will be True if the compound VIOLATES that rule
df["viol_mw"]       = df["molecular_weight"] > 500
df["viol_logp"]     = df["xlogp"] > 5
df["viol_hbd"]      = df["h_bond_donors"] > 5
df["viol_hba"]      = df["h_bond_acceptors"] > 10
df["viol_tpsa"]     = df["tpsa"] > 140
df["viol_rotbonds"] = df["rotatable_bonds"] > 10

# Count total violations per compound
violation_cols = ["viol_mw", "viol_logp", "viol_hbd",
                  "viol_hba", "viol_tpsa", "viol_rotbonds"]
df["ro5_violations"] = df[violation_cols].sum(axis=1)

# Flag compounds that pass Ro5 (0 or 1 violations allowed)
df["passes_ro5"] = df["ro5_violations"] <= 1

print(f"Compounds passing Ro5: {df['passes_ro5'].sum()} / {len(df)}")
print(f"\nViolation breakdown:")
print(df[violation_cols].sum())

Compounds passing Ro5: 78 / 80

Violation breakdown:
viol_mw          7
viol_logp        2
viol_hbd         0
viol_hba         0
viol_tpsa        0
viol_rotbonds    0
dtype: int64


In [12]:
# ============================================================
# 2d. ADMET properties via admet-ai
# ============================================================

try:
    from admet_ai import ADMETModel

    print("Loading ADMET-AI model (may take a moment)...")
    model = ADMETModel()

    print("Running predictions on 80 compounds...")
    smiles_list = df["isomeric_smiles"].tolist()
    admet_results = model.predict(smiles=smiles_list)

    print(f"ADMET results shape: {admet_results.shape}")
    print(f"\nAvailable ADMET properties:")
    print(list(admet_results.columns))

except Exception as e:
    print(f"ADMET-AI error: {e}")

Loading ADMET-AI model (may take a moment)...
Running predictions on 80 compounds...


model ensembles:   0%|                                    | 0/2 [00:00<?, ?it/s]GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/example3/lib/python3.14/site-packages/lightning/pytorch/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


/opt/anaconda3/envs/example3/lib/python3.14/site-packages/rich/live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/opt/anaconda3/envs/example3/lib/python3.14/site-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/opt/anaconda3/envs/example3/lib/python3.14/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=13` in the `DataLoader` to improve performance.


model ensembles:  50%|██████████████              | 1/2 [00:00<00:00,  4.11it/s]GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


model ensembles: 100%|████████████████████████████| 2/2 [00:00<00:00,  4.18it/s]

ADMET results shape: (80, 104)

Available ADMET properties:
['molecular_weight', 'logP', 'hydrogen_bond_acceptors', 'hydrogen_bond_donors', 'Lipinski', 'QED', 'stereo_centers', 'tpsa', 'PAINS_alert', 'BRENK_alert', 'NIH_alert', 'AMES', 'BBB_Martins', 'Bioavailability_Ma', 'CYP1A2_Veith', 'CYP2C19_Veith', 'CYP2C9_Substrate_CarbonMangels', 'CYP2C9_Veith', 'CYP2D6_Substrate_CarbonMangels', 'CYP2D6_Veith', 'CYP3A4_Substrate_CarbonMangels', 'CYP3A4_Veith', 'Carcinogens_Lagunin', 'ClinTox', 'DILI', 'HIA_Hou', 'NR-AR-LBD', 'NR-AR', 'NR-AhR', 'NR-Aromatase', 'NR-ER-LBD', 'NR-ER', 'NR-PPAR-gamma', 'PAMPA_NCATS', 'Pgp_Broccatelli', 'SR-ARE', 'SR-ATAD5', 'SR-HSE', 'SR-MMP', 'SR-p53', 'Skin_Reaction', 'hERG', 'Caco2_Wang', 'Clearance_Hepatocyte_AZ', 'Clearance_Microsome_AZ', 'Half_Life_Obach', 'HydrationFreeEnergy_FreeSolv', 'LD50_Zhu', 'Lipophilicity_AstraZeneca', 'PPBR_AZ', 'Solubility_AqSolDB', 'VDss_Lombardo', 'molecular_weight_drugbank_approved_percentile', 'logP_drugbank_approved_percentile'

In [13]:
## ============================================================
# Select relevant ADMET columns and merge into main DataFrame
# ============================================================

admet_cols = [
    "AMES",
    "BBB_Martins",
    "Bioavailability_Ma",
    "CYP2D6_Veith",
    "CYP3A4_Veith",
    "hERG",
    "HIA_Hou",
    "Caco2_Wang",
    "Solubility_AqSolDB",
    "DILI",
    "Pgp_Broccatelli",
    "BBB_Martins_drugbank_approved_percentile",
    "hERG_drugbank_approved_percentile",
]

admet_subset = admet_results[admet_cols].reset_index(drop=True)
df = df.reset_index(drop=True)
df = pd.concat([df, admet_subset], axis=1)

# Save this stage with a meaningful name
df_admet = df.copy()

print(f"df_admet has {len(df_admet)} compounds and {len(df_admet.columns)} columns.")
print(f"\nKey ADMET values for seed compounds:")
print(df_admet[["name", "BBB_Martins", "hERG", "AMES", "Bioavailability_Ma"]].head(4))

df_admet has 80 compounds and 36 columns.

Key ADMET values for seed compounds:
           name  BBB_Martins      hERG      AMES  Bioavailability_Ma
0  Daridorexant     0.668756  0.906027  0.140761            0.739615
1   Lemborexant     0.921039  0.730315  0.403990            0.968882
2    Suvorexant     0.847030  0.859074  0.049369            0.774514
3   Oveporexton     0.717796  0.885380  0.713080            0.734110


In [14]:
# ============================================================
# 2e. Composite drug-likeness score
# ============================================================

def score_mw(mw):
    """Ideal CNS molecular weight: 300-450 Da"""
    if 300 <= mw <= 450:   return 1.0
    if 200 <= mw < 300:    return 0.5
    if 450 < mw <= 550:    return 0.5
    return 0.0

def score_logp(lp):
    """Ideal CNS LogP: 1-4"""
    if pd.isna(lp):        return np.nan
    if 1.0 <= lp <= 4.0:   return 1.0
    if 0.0 <= lp < 1.0:    return 0.5
    if 4.0 < lp <= 5.0:    return 0.5
    return 0.0

def score_tpsa(t):
    """CNS drugs need low polarity: under 70 Å² ideal"""
    if pd.isna(t):         return np.nan
    if t <= 70:            return 1.0
    if t <= 90:            return 0.7
    if t <= 120:           return 0.4
    return 0.1

df_admet["mw_score"]   = df_admet["molecular_weight"].apply(score_mw)
df_admet["logp_score"] = df_admet["xlogp"].apply(score_logp)
df_admet["tpsa_score"] = df_admet["tpsa"].apply(score_tpsa)

# Composite score 0-3: higher = more CNS drug-like
df_admet["druglikeness_score"] = df_admet[["mw_score",
                                            "logp_score",
                                            "tpsa_score"]].sum(axis=1)

print("Drug-likeness scores for seed compounds:")
print(df_admet[["name", "mw_score", "logp_score",
                 "tpsa_score", "druglikeness_score"]].head(4))

print(f"\nScore distribution across all 80 compounds:")
print(df_admet["druglikeness_score"].describe().round(2))

Drug-likeness scores for seed compounds:
           name  mw_score  logp_score  tpsa_score  druglikeness_score
0  Daridorexant       0.5         0.5         0.7                 1.7
1   Lemborexant       1.0         1.0         0.7                 2.7
2    Suvorexant       0.5         0.5         0.7                 1.7
3   Oveporexton       0.5         1.0         0.4                 1.9

Score distribution across all 80 compounds:
count    80.00
mean      2.23
std       0.46
min       1.20
25%       1.90
50%       2.20
75%       2.70
max       2.70
Name: druglikeness_score, dtype: float64


In [15]:
# ============================================================
# 2f. Save the cleaned annotated library
# ============================================================

df_clean = df_admet.copy()

df_clean.to_csv("orexin_ligand_library_clean.csv", index=False)
print(f"Saved: orexin_ligand_library_clean.csv")
print(f"Final library: {len(df_clean)} compounds, {len(df_clean.columns)} columns")

print(f"\nSummary by drug class:")
print(df_clean.groupby("drug_class")[["molecular_weight", "xlogp",
                                       "ro5_violations",
                                       "druglikeness_score"]].mean().round(2))

Saved: orexin_ligand_library_clean.csv
Final library: 80 compounds, 40 columns

Summary by drug class:
                molecular_weight  xlogp  ro5_violations  druglikeness_score
drug_class                                                                 
DORA                      437.40   4.07            0.00                2.03
DORA_analog               443.68   3.67            0.14                2.25
agonist                   520.50   3.60            1.00                1.90
agonist_analog            456.03   3.60            0.00                2.21
